In [65]:
import pandas as pd
import jupyter_black

jupyter_black.load()
from pathlib import Path

out_dir = Path("data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

In [66]:
trans_df = pd.read_csv("data/raw/allTransactionsSorted_2024-25.csv")
trans_df.columns = trans_df.columns.str.lower().str.replace(" ", "_")

In [67]:
trans_df.dtypes

transaction_date    object
order_total         object
commission          object
merchant            object
domain              object
status              object
code                object
title               object
locked              object
payment_date        object
dtype: object

In [68]:
trans_df["commission"] = (
    trans_df["commission"].str.replace("$", "").str.replace(",", "").astype(float)
)

In [69]:
# Sorted desc by transaction count (grouped by merchant/item)

top_trans_count = (
    trans_df.groupby(["merchant", "title"])
    .agg({"transaction_date": "size", "commission": "sum"})
    .reset_index()
    .sort_values("transaction_date", ascending=False)
    .reset_index(drop=True)
    .rename(columns={"transaction_date": "transaction_count"})
    .head(10)
)
top_trans_count

,merchant,title,transaction_count,commission
0,Target,3pc Waffle Kitchen Towels - Figmint™,25,66.76
1,Target,Terra by Battat – Remote Control Infrared Ligh...,23,100.86
2,Target,Mondo Llama : Kids’ Crafts,21,73.36
3,Target,Target Anti-Theft Fanny Pack,13,25.34
4,Maelove,Glow Maker Vitamin C Serum,11,146.38
5,Geometry,Geometry Kitchen Tea Towels,9,78.42
6,Jouer Cosmetics,Skin Barrier Cream,9,73.65
7,Rèphr,RÈPHR | Intense Hydration Cream 1.0,8,39.49
8,Geometry LLC,Geometry Kitchen Tea Towels,8,44.36
9,Coolibar,Unisex Gannett UV Gloves | Sleek Grey,7,37.25


In [70]:
# Sorted desc by transactions dollar amount (grouped by merchant/item)

top_trans_sum = (
    trans_df.groupby(["merchant", "title"])
    .agg({"transaction_date": "size", "commission": "sum"})
    .reset_index()
    .sort_values("commission", ascending=False)
    .reset_index(drop=True)
    .rename(columns={"transaction_date": "transaction_count"})
    .head(10)
)
top_trans_sum

,merchant,title,transaction_count,commission
0,Cuyana,Travel Jewelry Case,5,235.20
1,Maelove,Glow Maker Vitamin C Serum,11,146.38
2,Target,Terra by Battat – Remote Control Infrared Ligh...,23,100.86
3,Geometry,Geometry Kitchen Tea Towels,9,78.42
4,Medik8,MEDIK8 | Advanced Night Ceramide,5,74.92
5,Jouer Cosmetics,Skin Barrier Cream,9,73.65
6,Target,Mondo Llama : Kids’ Crafts,21,73.36
7,Target,3pc Waffle Kitchen Towels - Figmint™,25,66.76
8,Clare V.,Clare V Fanny Pack,1,56.74
9,Saint Jane Beauty,Luxury Sun Ritual - Pore Smoothing SPF 30 Suns...,2,55.93


In [71]:
# Sorted desc by transactions mean dollar amount (grouped by merchant/item)

top_trans_mean = (
    trans_df.groupby(["merchant", "title"])
    .agg({"transaction_date": "size", "commission": "mean"})
    .reset_index()
    .sort_values("commission", ascending=False)
    .reset_index(drop=True)
    .rename(columns={"transaction_date": "transaction_count"})
    .head(10)
)
top_trans_mean

,merchant,title,transaction_count,commission
0,Clare V.,Clare V Fanny Pack,1,56.740
1,Boden USA,"Sienna Cotton Shirt-Ivory, Tennis",1,49.800
2,Cuyana,Travel Jewelry Case,5,47.040
3,Sephora,ORIGINAL Beautyblender Makeup Sponge,1,33.170
4,Monastery,Monastery | Healing Botanical Skincare,1,28.440
5,Saint Jane Beauty,Luxury Sun Ritual - Pore Smoothing SPF 30 Suns...,2,27.965
6,Medik8,Liquid Peptides™,1,27.450
7,LARQ,LARQ Bottle PureVis™,1,24.990
8,Tower 28,TOWER 28 | BeachPlease Luminous Tinted Cream B...,1,23.120
9,KYPRIS Beauty,Beauty Elixir IIi: Prismatic Array - Gentle Mo...,1,19.580


In [72]:
dfs = [top_trans_count, top_trans_sum, top_trans_mean]
names = ["top_trans_count", "top_trans_sum", "top_trans_mean"]

for name, df in zip(names, dfs):
    df.to_csv(out_dir / f"{name}.csv", index=False)